# Demand Forecasting & Inventory Planning Decision Dashboard

This notebook recreates the Excel workflow with Python, DuckDB, and pandas. It loads raw Olist CSV files, builds a delivered order-item mart, creates a complete Category x Month demand time series, applies three simple forecasting methods, scores the TEST period, and exports clean CSVs for downstream analysis.

The notebook is Colab-ready. In Colab, place the raw CSVs in `/content/data`. For local testing, keep the same CSVs in a project-level `data/` folder.

## 1. Install and Import Packages

DuckDB handles the SQL joins directly over CSV files. pandas and numpy handle the monthly grid, forecast calculations, summary flags, and exports.

In [ ]:
%pip install -q duckdb pandas numpy

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Configure Paths and Inputs

The path logic checks for `/content/data` first for Google Colab. If that folder does not exist, it falls back to a local `data/` folder. Exports are written to `outputs/`.

In [ ]:
COLAB_DATA_DIR = Path('/content/data')
LOCAL_DATA_DIR = Path('data')

DATA_DIR = COLAB_DATA_DIR if COLAB_DATA_DIR.exists() else LOCAL_DATA_DIR
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_FILES = {
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
    'customers': 'olist_customers_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
}

missing_files = [name for name in CSV_FILES.values() if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f'Missing raw CSV files in {DATA_DIR.resolve()}: {missing_files}. '
        'In Colab, upload them to /content/data. Locally, place them in data/.'
    )

print(f'Using data folder: {DATA_DIR.resolve()}')
print(f'Using output folder: {OUTPUT_DIR.resolve()}')

## 3. Define Business Scope

The dashboard uses delivered orders from January 2017 through August 2018. Forecasting is done at the Category x Month grain for 10 selected English category names.

In [ ]:
SELECTED_CATEGORIES = [
    'bed_bath_table',
    'health_beauty',
    'sports_leisure',
    'furniture_decor',
    'housewares',
    'telephony',
    'cool_stuff',
    'perfumery',
    'baby',
    'toys',
]

START_MONTH = pd.Timestamp('2017-01-01')
END_MONTH = pd.Timestamp('2018-08-01')

# The last six months are held out for forecast testing.
TEST_START_MONTH = pd.Timestamp('2018-03-01')

MONTHS = pd.date_range(START_MONTH, END_MONTH, freq='MS')
EXPECTED_TS_ROWS = len(SELECTED_CATEGORIES) * len(MONTHS)

print(f'Selected categories: {len(SELECTED_CATEGORIES)}')
print(f'Months in scope: {len(MONTHS)} ({START_MONTH:%Y-%m} to {END_MONTH:%Y-%m})')
print(f'Expected time-series rows: {EXPECTED_TS_ROWS}')

## 4. Build Delivered Order-Item Mart with DuckDB

This mart joins orders, order items, products, category translations, customers, and sellers. Each row represents one delivered order item in scope. It also adds operational fields used later in the dashboard: processing time, delivery time, total lead time, on-time delivery, freight ratio, and same-state fulfillment.

In [ ]:
def csv_path(file_key: str) -> str:
    """Return a DuckDB-friendly path string for one raw CSV file."""
    return (DATA_DIR / CSV_FILES[file_key]).as_posix()


con = duckdb.connect(database=':memory:')

# Raw Olist files are often ISO-like, but local exports may use M/D/YYYY H:MM.
# This macro keeps the notebook portable across both timestamp styles.
con.execute(
    """
    CREATE OR REPLACE MACRO parse_olist_ts(ts_value) AS (
        COALESCE(
            TRY_CAST(ts_value AS TIMESTAMP),
            TRY_STRPTIME(CAST(ts_value AS VARCHAR), '%m/%d/%Y %H:%M'),
            TRY_STRPTIME(CAST(ts_value AS VARCHAR), '%m/%d/%Y %H:%M:%S'),
            TRY_STRPTIME(CAST(ts_value AS VARCHAR), '%Y-%m-%d %H:%M:%S')
        )
    );
    """
)

category_sql = ', '.join([f"'{category}'" for category in SELECTED_CATEGORIES])

mart_sql = f"""
CREATE OR REPLACE TABLE delivered_order_item_mart AS
WITH joined AS (
    SELECT
        o.order_id,
        oi.order_item_id,
        o.customer_id,
        c.customer_unique_id,
        oi.product_id,
        oi.seller_id,
        COALESCE(t.product_category_name_english, p.product_category_name) AS category,
        c.customer_state,
        s.seller_state,
        parse_olist_ts(o.order_purchase_timestamp) AS purchase_ts,
        parse_olist_ts(o.order_approved_at) AS approved_ts,
        parse_olist_ts(o.order_delivered_carrier_date) AS delivered_carrier_ts,
        parse_olist_ts(o.order_delivered_customer_date) AS delivered_customer_ts,
        parse_olist_ts(o.order_estimated_delivery_date) AS estimated_delivery_ts,
        parse_olist_ts(oi.shipping_limit_date) AS shipping_limit_ts,
        CAST(oi.price AS DOUBLE) AS price,
        CAST(oi.freight_value AS DOUBLE) AS freight_value
    FROM read_csv_auto('{csv_path('orders')}', HEADER=TRUE) AS o
    INNER JOIN read_csv_auto('{csv_path('order_items')}', HEADER=TRUE) AS oi
        ON o.order_id = oi.order_id
    LEFT JOIN read_csv_auto('{csv_path('products')}', HEADER=TRUE) AS p
        ON oi.product_id = p.product_id
    LEFT JOIN read_csv_auto('{csv_path('category_translation')}', HEADER=TRUE) AS t
        ON p.product_category_name = t.product_category_name
    LEFT JOIN read_csv_auto('{csv_path('customers')}', HEADER=TRUE) AS c
        ON o.customer_id = c.customer_id
    LEFT JOIN read_csv_auto('{csv_path('sellers')}', HEADER=TRUE) AS s
        ON oi.seller_id = s.seller_id
    WHERE o.order_status = 'delivered'
      AND parse_olist_ts(o.order_purchase_timestamp) >= TIMESTAMP '2017-01-01'
      AND parse_olist_ts(o.order_purchase_timestamp) < TIMESTAMP '2018-09-01'
      AND COALESCE(t.product_category_name_english, p.product_category_name) IN ({category_sql})
      AND o.order_purchase_timestamp IS NOT NULL
      AND o.order_delivered_carrier_date IS NOT NULL
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
)
SELECT
    *,
    CAST(date_trunc('month', purchase_ts) AS DATE) AS order_month,
    date_diff('day', purchase_ts, delivered_carrier_ts) AS processing_days,
    date_diff('day', delivered_carrier_ts, delivered_customer_ts) AS delivery_days,
    date_diff('day', purchase_ts, delivered_customer_ts) AS total_lead_time_days,
    CASE WHEN delivered_customer_ts <= estimated_delivery_ts THEN 1 ELSE 0 END AS delivered_on_time_flag,
    CASE WHEN customer_state = seller_state THEN 1 ELSE 0 END AS same_state_flag,
    freight_value / NULLIF(price + freight_value, 0) AS item_freight_ratio
FROM joined;
"""

con.execute(mart_sql)

mart_row_count = con.execute('SELECT COUNT(*) FROM delivered_order_item_mart').fetchone()[0]
print(f'Delivered order-item mart rows: {mart_row_count:,}')

## 5. Create Operational Metrics

Operational metrics are summarized by category. These features help translate forecast results into inventory planning recommendations.

In [ ]:
ops_metrics = con.execute(
    """
    SELECT
        category,
        COUNT(*) AS item_count,
        COUNT(DISTINCT order_id) AS order_count,
        ROUND(AVG(processing_days), 2) AS avg_processing_days,
        ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
        ROUND(AVG(total_lead_time_days), 2) AS avg_total_lead_time,
        ROUND(STDDEV_SAMP(total_lead_time_days), 2) AS lead_time_stddev,
        ROUND(AVG(delivered_on_time_flag), 4) AS otd_rate,
        ROUND(SUM(freight_value) / NULLIF(SUM(price + freight_value), 0), 4) AS freight_ratio,
        ROUND(AVG(same_state_flag), 4) AS same_state_rate
    FROM delivered_order_item_mart
    GROUP BY category
    ORDER BY category
    """
).df()

ops_metrics

## 6. Build a Complete Category x Month Time Series

This step creates the required 10 categories x 20 months grid. Months with no observed demand are kept and filled with zero demand, which avoids accidental gaps in the forecasting timeline.

In [ ]:
monthly_actuals = con.execute(
    """
    SELECT
        category,
        order_month,
        COUNT(*) AS actual_demand,
        COUNT(DISTINCT order_id) AS order_count,
        ROUND(SUM(price), 2) AS revenue,
        ROUND(SUM(freight_value), 2) AS freight_value
    FROM delivered_order_item_mart
    GROUP BY category, order_month
    """
).df()

monthly_actuals['order_month'] = pd.to_datetime(monthly_actuals['order_month'])

complete_grid = pd.MultiIndex.from_product(
    [SELECTED_CATEGORIES, MONTHS],
    names=['category', 'order_month']
).to_frame(index=False)

timeseries_full = complete_grid.merge(
    monthly_actuals,
    on=['category', 'order_month'],
    how='left'
)

fill_zero_cols = ['actual_demand', 'order_count', 'revenue', 'freight_value']
timeseries_full[fill_zero_cols] = timeseries_full[fill_zero_cols].fillna(0)
timeseries_full[fill_zero_cols] = timeseries_full[fill_zero_cols].astype({
    'actual_demand': 'int64',
    'order_count': 'int64',
    'revenue': 'float64',
    'freight_value': 'float64',
})

timeseries_full['period_type'] = np.where(
    timeseries_full['order_month'] >= TEST_START_MONTH,
    'TEST',
    'TRAIN'
)

timeseries_full = timeseries_full.sort_values(['category', 'order_month']).reset_index(drop=True)

print(f'Time-series rows: {len(timeseries_full):,}')
timeseries_full.head(12)

## 7. Apply Forecasting Methods

The notebook applies three simple, Excel-friendly forecast methods for each category:

- Moving Average with a 3-month window
- Weighted Moving Average with weights 0.5, 0.3, and 0.2, where the most recent month gets the largest weight
- Exponential Smoothing with alpha = 0.3

Each month's forecast uses only prior months' actual demand.

In [ ]:
def add_forecasts(category_frame: pd.DataFrame) -> pd.DataFrame:
    """Add one-step-ahead forecasts for a single category."""
    frame = category_frame.sort_values('order_month').copy()
    demand = frame['actual_demand'].astype(float).reset_index(drop=True)

    # Simple 3-month moving average. Early months use available prior history.
    frame['forecast_ma_3'] = demand.shift(1).rolling(window=3, min_periods=1).mean().to_numpy()

    # Weighted moving average: most recent month = 0.5, then 0.3, then 0.2.
    weights = np.array([0.5, 0.3, 0.2])
    weighted_forecasts = []
    for idx in range(len(demand)):
        recent_history = demand.iloc[max(0, idx - 3):idx].iloc[::-1].to_numpy(dtype=float)
        if len(recent_history) == 0:
            weighted_forecasts.append(np.nan)
        else:
            active_weights = weights[:len(recent_history)]
            weighted_forecasts.append(np.dot(recent_history, active_weights) / active_weights.sum())
    frame['forecast_wma_3'] = weighted_forecasts

    # Exponential smoothing with alpha = 0.3.
    alpha = 0.3
    exp_smooth_forecasts = [np.nan] * len(demand)
    if len(demand) > 1:
        next_forecast = demand.iloc[0]
        for idx in range(1, len(demand)):
            exp_smooth_forecasts[idx] = next_forecast
            next_forecast = alpha * demand.iloc[idx] + (1 - alpha) * next_forecast
    frame['forecast_exp_smooth_alpha_0_3'] = exp_smooth_forecasts

    return frame


forecast_frames = []
for category, category_frame in timeseries_full.groupby('category', sort=False):
    forecast_frames.append(add_forecasts(category_frame))

timeseries_full = pd.concat(forecast_frames, ignore_index=True)

forecast_columns = ['forecast_ma_3', 'forecast_wma_3', 'forecast_exp_smooth_alpha_0_3']
timeseries_full[['category', 'order_month', 'actual_demand', 'period_type'] + forecast_columns].head(12)

## 8. Calculate TEST-Period Error Metrics

Error metrics are calculated only on TEST months. Bias is defined as `actual demand - forecast`, so positive bias means the forecast was too low, or under-forecasting. Negative bias means the forecast was too high, or over-forecasting.

In [ ]:
MODEL_MAP = {
    'forecast_ma_3': 'Moving Average 3',
    'forecast_wma_3': 'Weighted Moving Average 0.5/0.3/0.2',
    'forecast_exp_smooth_alpha_0_3': 'Exponential Smoothing alpha 0.3',
}

test_rows = timeseries_full[timeseries_full['period_type'] == 'TEST'].copy()

error_records = []
for category in SELECTED_CATEGORIES:
    category_test = test_rows[test_rows['category'] == category].copy()
    actual = category_test['actual_demand'].astype(float)
    for forecast_col, model_name in MODEL_MAP.items():
        forecast = category_test[forecast_col].astype(float)
        error = actual - forecast
        absolute_error = error.abs()
        absolute_percentage_error = np.where(actual == 0, np.nan, absolute_error / actual)

        error_records.append({
            'category': category,
            'model': model_name,
            'test_months': len(category_test),
            'mae': absolute_error.mean(),
            'mape': np.nanmean(absolute_percentage_error),
            'bias': error.mean(),
            'bias_interpretation': np.where(
                error.mean() > 0,
                'under-forecast',
                np.where(error.mean() < 0, 'over-forecast', 'neutral')
            ).item(),
        })

error_metrics = pd.DataFrame(error_records)
error_metrics[['mae', 'mape', 'bias']] = error_metrics[['mae', 'mape', 'bias']].round(4)

error_metrics.sort_values(['category', 'mae']).head(15)

## 9. Build Forecast Summary and Decision Flags

This summary picks the best model by TEST-period MAE for each category, then adds planning-friendly labels: demand class, forecast risk, bias concern, operational risk, implied safety buffer, and final recommendation.

In [ ]:
best_idx = error_metrics.groupby('category')['mae'].idxmin()
best_model = (
    error_metrics.loc[best_idx, ['category', 'model', 'mae', 'mape', 'bias']]
    .rename(columns={
        'model': 'best_by_mae',
        'mae': 'best_mae',
        'mape': 'best_mape',
        'bias': 'best_bias',
    })
)

demand_profile = (
    timeseries_full
    .groupby('category', as_index=False)
    .agg(
        avg_monthly_demand=('actual_demand', 'mean'),
        avg_test_monthly_demand=('actual_demand', lambda x: x[timeseries_full.loc[x.index, 'period_type'].eq('TEST')].mean()),
        total_demand=('actual_demand', 'sum'),
        zero_demand_months=('actual_demand', lambda x: int((x == 0).sum())),
    )
)

q33 = demand_profile['avg_monthly_demand'].quantile(0.33)
q66 = demand_profile['avg_monthly_demand'].quantile(0.66)

def classify_demand(avg_monthly_demand: float) -> str:
    if avg_monthly_demand >= q66:
        return 'High'
    if avg_monthly_demand >= q33:
        return 'Medium'
    return 'Low'


def classify_forecast_risk(mape: float) -> str:
    if pd.isna(mape):
        return 'High'
    if mape <= 0.15:
        return 'Low'
    if mape <= 0.30:
        return 'Medium'
    return 'High'


def classify_operational_risk(row: pd.Series) -> str:
    if (
        row['otd_rate'] < 0.85
        or row['lead_time_stddev'] > 8
        or row['avg_total_lead_time'] > 18
        or row['freight_ratio'] > 0.20
    ):
        return 'High'
    if (
        row['otd_rate'] < 0.92
        or row['lead_time_stddev'] > 5
        or row['avg_total_lead_time'] > 14
        or row['freight_ratio'] > 0.15
    ):
        return 'Medium'
    return 'Low'


forecast_summary = (
    demand_profile
    .merge(best_model, on='category', how='left')
    .merge(ops_metrics, on='category', how='left')
)

forecast_summary['demand_class'] = forecast_summary['avg_monthly_demand'].apply(classify_demand)
forecast_summary['forecast_risk'] = forecast_summary['best_mape'].apply(classify_forecast_risk)

# A bias concern means the average monthly miss is material relative to TEST demand.
bias_threshold = np.maximum(2, forecast_summary['avg_test_monthly_demand'] * 0.10)
forecast_summary['bias_concern_flag'] = forecast_summary['best_bias'].abs() >= bias_threshold

forecast_summary['operational_risk'] = forecast_summary.apply(classify_operational_risk, axis=1)

risk_band = {
    'Low': (0.5, 1.0),
    'Medium': (1.0, 1.5),
    'High': (1.5, 2.0),
}
ops_factor = {'Low': 1.0, 'Medium': 1.25, 'High': 1.5}

def safety_buffer(row: pd.Series, side: str) -> int:
    min_multiplier, max_multiplier = risk_band[row['forecast_risk']]
    forecast_multiplier = min_multiplier if side == 'min' else max_multiplier
    lead_time_stddev = row['lead_time_stddev'] if pd.notna(row['lead_time_stddev']) else 1
    lead_time_stddev = max(lead_time_stddev, 1)
    avg_daily_demand = row['avg_monthly_demand'] / 30
    buffer_units = avg_daily_demand * lead_time_stddev * forecast_multiplier * ops_factor[row['operational_risk']]
    return int(np.ceil(max(1, buffer_units)))


forecast_summary['implied_min_safety_buffer'] = forecast_summary.apply(lambda row: safety_buffer(row, 'min'), axis=1)
forecast_summary['implied_max_safety_buffer'] = forecast_summary.apply(lambda row: safety_buffer(row, 'max'), axis=1)

def recommendation(row: pd.Series) -> str:
    buffer_text = f"Carry about {row['implied_min_safety_buffer']}-{row['implied_max_safety_buffer']} units as a minimum planning buffer."
    if row['best_bias'] > 0 and row['bias_concern_flag']:
        bias_text = 'Best model is under-forecasting; lift the planning forecast or add buffer before stockouts appear.'
    elif row['best_bias'] < 0 and row['bias_concern_flag']:
        bias_text = 'Best model is over-forecasting; watch excess stock and review reorder quantities.'
    else:
        bias_text = 'Forecast bias is not material in the TEST period.'

    if row['forecast_risk'] == 'High' or row['operational_risk'] == 'High':
        return f"High-touch weekly review. {buffer_text} {bias_text}"
    if row['forecast_risk'] == 'Medium' or row['operational_risk'] == 'Medium':
        return f"Standard weekly review. {buffer_text} {bias_text}"
    return f"Stable monthly review. {buffer_text} {bias_text}"


forecast_summary['final_recommendation'] = forecast_summary.apply(recommendation, axis=1)

summary_columns = [
    'category',
    'demand_class',
    'forecast_risk',
    'best_by_mae',
    'best_mae',
    'best_mape',
    'best_bias',
    'bias_concern_flag',
    'operational_risk',
    'implied_min_safety_buffer',
    'implied_max_safety_buffer',
    'final_recommendation',
]

forecast_summary = forecast_summary[summary_columns].sort_values('category').reset_index(drop=True)
forecast_summary

## 10. Create Quality Checks

The quality check table makes the key row-count and completeness checks explicit. The final cell also prints them for quick notebook review.

In [ ]:
ts_quality_check = pd.DataFrame([
    {
        'check_name': 'mart_row_count_positive',
        'expected_value': '> 0',
        'actual_value': mart_row_count,
        'passed': mart_row_count > 0,
    },
    {
        'check_name': 'timeseries_row_count',
        'expected_value': EXPECTED_TS_ROWS,
        'actual_value': len(timeseries_full),
        'passed': len(timeseries_full) == EXPECTED_TS_ROWS,
    },
    {
        'check_name': 'forecast_summary_row_count',
        'expected_value': len(SELECTED_CATEGORIES),
        'actual_value': len(forecast_summary),
        'passed': len(forecast_summary) == len(SELECTED_CATEGORIES),
    },
    {
        'check_name': 'no_missing_final_recommendation',
        'expected_value': 0,
        'actual_value': int(forecast_summary['final_recommendation'].isna().sum()),
        'passed': forecast_summary['final_recommendation'].notna().all(),
    },
    {
        'check_name': 'selected_categories_present',
        'expected_value': len(SELECTED_CATEGORIES),
        'actual_value': timeseries_full['category'].nunique(),
        'passed': timeseries_full['category'].nunique() == len(SELECTED_CATEGORIES),
    },
])

ts_quality_check

## 11. Export Clean CSV Outputs

These files are the Python equivalents of the cleaned Excel workflow outputs. They can be opened directly in Excel or used by another BI/reporting tool later.

In [ ]:
export_paths = {
    'timeseries_full': OUTPUT_DIR / 'timeseries_full.csv',
    'ops_metrics': OUTPUT_DIR / 'ops_metrics.csv',
    'forecast_summary': OUTPUT_DIR / 'forecast_summary.csv',
    'error_metrics': OUTPUT_DIR / 'error_metrics.csv',
    'ts_quality_check': OUTPUT_DIR / 'ts_quality_check.csv',
}

timeseries_export = timeseries_full.copy()
timeseries_export['order_month'] = timeseries_export['order_month'].dt.strftime('%Y-%m-%d')

timeseries_export.to_csv(export_paths['timeseries_full'], index=False)
ops_metrics.to_csv(export_paths['ops_metrics'], index=False)
forecast_summary.to_csv(export_paths['forecast_summary'], index=False)
error_metrics.to_csv(export_paths['error_metrics'], index=False)
ts_quality_check.to_csv(export_paths['ts_quality_check'], index=False)

print('QA CHECKS')
print(f"Mart row count: {mart_row_count:,}")
print(f"Timeseries row count: {len(timeseries_full):,} (should be {EXPECTED_TS_ROWS})")
print(f"Forecast summary row count: {len(forecast_summary):,} (should be {len(SELECTED_CATEGORIES)})")
print(f"Missing final_recommendation values: {forecast_summary['final_recommendation'].isna().sum()}")
print('\nExported files:')
for name, path in export_paths.items():
    print(f"- {name}: {path.resolve()}")